In [41]:
import json
import os
from tqdm import tqdm, trange

def read_json_file(json_path):
    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

数据格式：
{'begin': 359,
 'end': 370,
 'label': 'variable',
 'substring': 'temperature',
 'identifier': 'https://gcmd.earthdata.nasa.gov/kms/concept/3e7854b1-d7bc-4e5b-af2f-57d8d7bc5648'}

In [42]:
from nltk.tokenize import sent_tokenize

def split_into_sentences(text):
    sentences = sent_tokenize(text)  # 使用nltk进行句子切分
    return sentences
def get_sentence_positions(text):
    sentences = split_into_sentences(text)
    positions = []
    start = 0
    for sentence in sentences:
        start_idx = text.find(sentence, start)  # 找到句子的起始位置
        end_idx = start_idx + len(sentence)
        positions.append((start_idx, end_idx))
        start = end_idx
    return sentences, positions
# 为了适应RoBERTa的输入最大长度，我们需要对文本进行切分，然后每个chunk包含多个句子，总的长度不超过max_length。即总的长度不能超过1500
# 包含至少3个句子。然后我们要把原始的实体位置映射到新的chunk上。
def get_chunks(sentences, positions, max_length=1536):
    chunks = []
    chunk = []
    chunk_positions = []
    chunk_length = 0
    for sentence, position in zip(sentences, positions):
        sentence_length = len(sentence)
        if chunk_length + sentence_length < max_length:
            chunk.append(sentence)
            chunk_positions.append(position)
            chunk_length += sentence_length
        else:
            chunks.append((chunk, chunk_positions))
            chunk = [sentence]
            chunk_positions = [position]
            chunk_length = sentence_length
    if chunk:
        chunks.append((chunk, chunk_positions))
    return chunks
def find_entities(chunk_start, chunk_end, entities):
    selected_entities = []
    for entity in entities:
        entity_start = entity["begin"]
        entity_end = entity["end"]
        if entity_start >= chunk_start and entity_end <= chunk_end:
            selected_entities.append(entity)
    return selected_entities

def update_chunk_entity_spans(start_position, entities):
    updated_entities = []
    for entity in entities:
        entity["begin"] = entity["begin"] - start_position
        entity["end"] = entity["end"] - start_position
        updated_entities.append(entity)
    return updated_entities

In [43]:
files = os.listdir("./datasets/climate/")
for file_name in tqdm(files):
    data = read_json_file(f"./datasets/climate/{file_name}")
    # process one document
    entities = data["entities"]
    entities = [value for key, value in entities.items()]
    text = data["text"]
    # split the text into chunks
    sentences, positions = get_sentence_positions(text)
    chunks = get_chunks(sentences, positions)
    # process each chunk
    # 遍历每个chunk，找到实体在chunk中的位置，然后映射到原始文本中的位置。更新实体的位置。
    # 1. 先找到一个chunk中所有的实体
    doc = []
    for chunk in chunks:
        end_position = chunk[1][-1][1]
        start_position = chunk[1][0][0]
        entities_in_chunk = find_entities(start_position, end_position, entities)
        chunk_entities = update_chunk_entity_spans(start_position, entities_in_chunk)
        chunk_span = (start_position, end_position)
        chunk_text = text[start_position:end_position]
        doc.append({
            "text": chunk_text,
            "span": chunk_span,
            "entities": chunk_entities,
        })
    # save the processed data
    # save doc to json
    with open(f"./datasets/climate_chunk/{file_name}", "w", encoding="utf-8") as file:
        json.dump(doc, file, ensure_ascii=False, indent=4)

100%|██████████| 25/25 [00:00<00:00, 66.94it/s]


In [44]:
chunked_files = os.listdir("./datasets/climate_chunk/")
for file in chunked_files:
    data = read_json_file(f"./datasets/climate_chunk/{file}")
    for chunk in data:
        text = chunk["text"]
        entities = chunk["entities"]
        for entity in entities:
            entity_text = text[entity["begin"]:entity["end"]]
            assert entity_text == entity["substring"], f"{entity_text} != {entity['substring']}"

In [45]:
chunked_files = os.listdir("./datasets/climate_chunk/")
print(len(chunked_files))

25


In [52]:
chunk_num = 0
entities_num = 0
ent_set = set()
ent_none = 0

for file in chunked_files:
    data = read_json_file(f"./datasets/climate_chunk/{file}")
    chunk_num += len(data)
    for chunk in data:
        text = chunk["text"]
        entities = chunk["entities"]
        entities_num += len(entities)
        for entity in entities:
            entity_text = text[entity["begin"]:entity["end"]]
            assert entity_text == entity["substring"], f"{entity_text} != {entity['substring']}"
            if entity["identifier"] is not None:
                ent_set.add(entity["identifier"])
            else:
                ent_none += 1

In [53]:
print(chunk_num)
print(entities_num)
print(len(ent_set))
print(ent_none)

848
12222
925
2164
